# GrowBot — Domain-Randomized Walking Policy (RMA-style, non-privileged)

Trains a walking policy for the **real public-rollout body** (the MuJoCo model built from `NEWBODY.stl` / `NEWLEG.stl`) that transfers to hardware despite the ways real builds differ from each other.

**Just press `Runtime → Run all`.** (You'll get one Google-Drive auth popup if you allow checkpoint saving; decline it and it falls back to local storage.)

---
### What this does
Building on the analysis of the earlier `DR_25hz` run (heavy domain randomization + action history + high entropy = a slow-to-train but *deployable* policy), this notebook adds the three randomizations you asked for and keeps the observation **non-privileged** (only what the real robot actually senses):

* **Robot mass** — alkaline vs lithium AAs, different phones (`0.80–1.25×`, ~384–600 g).
* **Center of mass** — battery fore/aft/height mounting tolerance (`±30 mm` X, `±15 mm` Y, `-10..+15 mm` Z).
* **Leg length** — print/material variation shared across both legs (`0.85–1.15×`) plus small per-leg assembly asymmetry (`±4 %`). *This actually rescales the leg geometry, mass & inertia in the physics — verified to change contact height, not just a label.*
* Plus the `DR_25hz` set: servo gain, friction, actuator slew + lag, the phone→cloud→Pico relay delay + packet-drops, and IMU misalignment.

### How the body latents are handled (RMA-style)
Full two-phase RMA (train a privileged encoder, then distil a history adapter) is fragile to run unattended in Brax. Instead this uses the robust, Brax-native equivalent:

* **Policy (what deploys to the phone) — non-privileged.** It sees only a stacked history of `[roll, pitch, yaw, gyro, last action]`. That temporal window is what lets it *implicitly* infer the body it's driving (a long leg, a rear CoM and a heavy battery all leave a signature in how the body responds) — the same job RMA's adaptation module does, learned concurrently instead of by distillation.
* **Critic (thrown away at deployment) — privileged.** The value function additionally sees the true latents (mass, CoM, leg scales, gains, delay…) via Brax's `value_obs_key`. A privileged critic cuts the value-estimation noise that heavy randomization otherwise injects — this is the part of RMA that most improves training, kept without any of its fragility.

Net result: a policy that never sees privileged info (so it runs on the real robot unchanged) but trains against a critic that does.

### Reading the logs (open the TensorBoard below during/after training)
Every quantity is logged so a failure mode is visible at a glance — see the **"How to read these logs"** section at the very bottom for the full table. Quick version:
`Diagnostics/FallRate` high → losing balance · `Diagnostics/FwdVelocity_mps` ≈ 0 → too timid / underpowered · `< 0` → walking backward · `Diagnostics/ActionSaturation` high → servo-limited under the mass range · `Train/ValueLoss` exploding → critic struggling with the DR spread.

## 1 · Install a pinned, mutually-compatible stack
The latest `brax` still calls a JAX API that the latest `jax` removed, so a fresh `pip install brax jax` breaks. These exact pins are verified to work together (and keep the asymmetric actor-critic API).

In [ ]:
# GPU build of the pinned stack (falls back to CPU automatically if no GPU).
!pip install -q "jax[cuda12]==0.4.36" "jaxlib==0.4.36" \
  "brax==0.12.1" "flax==0.10.2" "optax==0.2.4" "orbax-checkpoint==0.6.4" \
  "mujoco==3.2.7" "mujoco-mjx==3.2.7" "tensorboardX" 2>&1 | tail -3

# Load TensorBoard now so it's ready to watch training live.
%load_ext tensorboard

In [ ]:
import jax
print("JAX version:", jax.__version__)
# if not jax.__version__.startswith("0.4.35"):
#     print("\n[!] Wrong JAX version loaded (Colab pre-imported an old one).")
#     print("    Do: Runtime > Restart session, then Runtime > Run all again.")
if not jax.__version__.startswith("0.4.36"):
     print("\n[!] Wrong JAX version loaded (Colab pre-imported an old one).")
     print("    Do: Runtime > Restart session, then Runtime > Run all again.")
print("JAX devices:", jax.devices())
if jax.devices()[0].platform != "gpu":
    print("\n[!] No GPU detected — training will be SLOW.")
    print("    Set Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.")
else:
    print("\n[OK] GPU ready.")

## 2 · The robot model (MJCF)
The nominal public-rollout body, generated from the STLs by `build_mjcf.py`. Domain randomization rescales/reweights it per-episode at runtime; this string is just the baseline.

In [ ]:
MJCF_XML = r"""
<mujoco model="Growbot">
  <option gravity="0 0 -9.81" timestep="0.005" integrator="RK4" solver="CG" iterations="10" ls_iterations="10"/>

  <default>
    <geom friction="1.2 0.1 0.1" solref="0.005 1" solimp="0.99 0.99 0.01" condim="3"/>
    <joint damping="0.03" armature="0.002"/>
    <position kp="0.75" kv="0.05" ctrlrange="-1.57 1.57" forcerange="-1.5 1.5"/>
  </default>

  <worldbody>
    <light name="sun" pos="0 0 3" dir="0 0 -1" diffuse="0.8 0.8 0.8" specular="0.2 0.2 0.2" castshadow="true"/>
    <geom name="floor" type="plane" size="0 0 0.05" rgba="0.76 0.87 0.70 1"/>

    <body name="base_body" pos="0 0 0.3">
      <joint name="root_joint" type="free"/>
      <inertial pos="-0.02 0 0" mass="0.426606" diaginertia="0.000168019 0.00102813 0.00119471"/>
      <geom name="torso_geom" type="box" size="0.085 0.0343 0.00225" pos="0 0 0" rgba="0.7 0.7 0.7 1"/>
      <geom name="servo_rib" type="box" size="0.0163 0.0343 0.005" pos="0.00565 0 0.00725" contype="0" conaffinity="0" rgba="0.55 0.55 0.6 1"/>

      <body name="right_leg" pos="0.00565 -0.04505 0.00658863">
        <joint name="joint_1" type="hinge" axis="0 1 0" limited="true" range="-90 90"/>
        <geom name="lower_leg_1" type="box" size="0.0105 0.00675 0.0329144" pos="0.000894822 0 -0.0329144" mass="0.026697" rgba="0.2 0.2 0.8 1"/>
      </body>

      <body name="left_leg" pos="0.00565 0.04505 0.00658863">
        <joint name="joint_2" type="hinge" axis="0 1 0" limited="true" range="-90 90"/>
        <geom name="lower_leg_2" type="box" size="0.0105 0.00675 0.0329144" pos="0.000894822 0 -0.0329144" mass="0.026697" rgba="0.2 0.2 0.8 1"/>
      </body>
    </body>
  </worldbody>

  <actuator>
    <position name="servo_1" joint="joint_1"/>
    <position name="servo_2" joint="joint_2"/>
  </actuator>
</mujoco>
"""

## 3 · Configuration — all the knobs in one place
DR ranges are tied to real-build variation (see comments). Training hyper-parameters mirror the proven `DR_25hz` run; the network is a little larger because the policy now has a history to reason over.

In [ ]:
HISTORY_LEN = 10          # frames of proprio+action stacked for the (non-privileged) policy obs
FRAME_DIM = 8             # per frame: roll,pitch,yaw,gyro(3),last_action(2)
CTRL_HZ_NFRAMES = 8       # 25 Hz control (timestep 0.005 * 8 = 0.04s)

# ---- domain randomization ranges ----
MASS_SCALE = (0.80, 1.25)         # alkaline vs lithium AA, phone weight
DCOM_X = (-0.030, 0.030)          # battery fore/aft position tolerance (m)
DCOM_Y = (-0.015, 0.015)
DCOM_Z = (-0.010, 0.015)          # battery low vs phone high
LEG_SHARED = (0.85, 1.15)         # print/material length variation (both legs together)
LEG_ASYM = (0.96, 1.04)           # per-leg assembly/horn-seat asymmetry
GAIN_MULT = (0.75, 1.25)          # servo Kp/Kv spread
FRICTION = (0.6, 1.4)
SLEW = (8.7, 10.5)                # rad/s
TAU = (0.015, 0.035)              # actuator lag (s)
CLOUD_DELAY = (0.020, 0.090)      # phone->cloud->Pico relay (s)
IMU_OFFSET = 0.26                 # +/- rad IMU mount misalignment
STALL_PROB = 0.08                 # cloud packet-drop probability


# ---- PPO hyper-parameters (mirroring the successful DR_25hz run) ----
NUM_TIMESTEPS       = 60_000_000   # raise to 100M+ for a stronger policy if Colab stays connected
NUM_EVALS           = 60           # = number of TensorBoard points & checkpoints
EPISODE_LENGTH      = 1000
NUM_ENVS            = 2048
BATCH_SIZE          = 1024
NUM_MINIBATCHES     = 32
NUM_UPDATES_PER_BATCH = 4
UNROLL_LENGTH       = 20
DISCOUNTING         = 0.99
LEARNING_RATE       = 3e-4
ENTROPY_COST        = 0.05         # high entropy -> exploration, as in DR_25hz
REWARD_SCALING      = 1.0
POLICY_HIDDEN       = (128, 128)   # non-privileged policy (reasons over the history)
VALUE_HIDDEN        = (256, 256)   # privileged critic
SEED                = 0

## 4 · Environment
`reset()` draws a fresh random body every episode; `step()` applies the cloud-relay delay + servo slew/lag, steps the physics with that episode's randomized model, and emits a **dict observation**: `state` (non-privileged, for the policy) and `privileged` (latents, for the critic only).

In [ ]:
import jax
from jax import numpy as jnp
import numpy as np
import mujoco
from mujoco import mjx
from brax import envs
from brax.envs.base import PipelineEnv, State
import functools

class GrowbotRMAEnv(PipelineEnv):
    def __init__(self, **kwargs):
        mj_model = mujoco.MjModel.from_xml_string(MJCF_XML)
        mj_model.opt.solver = mujoco.mjtSolver.mjSOL_CG
        mj_model.opt.iterations = 6
        mj_model.opt.ls_iterations = 6

        # indices we need for domain randomization / rewards (static python ints)
        self._torso_bid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "base_body")
        self._legR_bid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "right_leg")
        self._legL_bid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "left_leg")
        self._legR_gid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "lower_leg_1")
        self._legL_gid = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "lower_leg_2")

        sys = mjx.put_model(mj_model)
        super().__init__(sys=sys, backend="mjx", n_frames=CTRL_HZ_NFRAMES, **kwargs)

        # nominal leg geometry (half-length along local Z, and the box half-sizes)
        self._leg_half_len0 = float(sys.geom_size[self._legR_gid, 2])
        self._leg_hx = float(sys.geom_size[self._legR_gid, 0])   # thickness/2
        self._leg_hy = float(sys.geom_size[self._legR_gid, 1])   # width/2
        self._leg_mass0 = float(sys.body_mass[self._legR_bid])
        self._torso_ipos0 = np.array(sys.body_ipos[self._torso_bid])

    @property
    def action_size(self):
        return self.sys.nu

    # ---------------- domain randomization ----------------
    def _randomize(self, rng):
        keys = jax.random.split(rng, 13)
        mass_scale = jax.random.uniform(keys[0], (), minval=MASS_SCALE[0], maxval=MASS_SCALE[1])
        # CoM offset around nominal (battery bottom-rear; phone top)
        dcom = jnp.array([
            jax.random.uniform(keys[1], (), minval=DCOM_X[0], maxval=DCOM_X[1]),
            jax.random.uniform(keys[2], (), minval=DCOM_Y[0], maxval=DCOM_Y[1]),
            jax.random.uniform(keys[3], (), minval=DCOM_Z[0], maxval=DCOM_Z[1]),
        ])
        # legs share a print-length scale, plus a small independent per-leg asymmetry
        leg_shared = jax.random.uniform(keys[4], (), minval=LEG_SHARED[0], maxval=LEG_SHARED[1])
        legR_scale = leg_shared * jax.random.uniform(keys[5], (), minval=LEG_ASYM[0], maxval=LEG_ASYM[1])
        legL_scale = leg_shared * jax.random.uniform(keys[12], (), minval=LEG_ASYM[0], maxval=LEG_ASYM[1])
        gain_mult = jax.random.uniform(keys[6], (), minval=GAIN_MULT[0], maxval=GAIN_MULT[1])
        fric = jax.random.uniform(keys[7], (self.sys.ngeom,), minval=FRICTION[0], maxval=FRICTION[1])
        slew = jax.random.uniform(keys[8], (2,), minval=SLEW[0], maxval=SLEW[1])
        tau = jax.random.uniform(keys[9], (2,), minval=TAU[0], maxval=TAU[1])
        cloud_delay = jax.random.uniform(keys[10], (), minval=CLOUD_DELAY[0], maxval=CLOUD_DELAY[1])
        imu_offset = jax.random.uniform(keys[11], (3,), minval=-IMU_OFFSET, maxval=IMU_OFFSET)

        # --- build randomized sys ---
        new_mass = self.sys.body_mass * mass_scale
        new_inertia = self.sys.body_inertia * mass_scale

        # per-leg geometry: scale length along local Z
        def scale_leg(sys_fields, gid, bid, s):
            gs, gp, bm, bi, bip = sys_fields
            half = self._leg_half_len0 * s
            gs = gs.at[gid, 2].set(half)
            gp = gp.at[gid, 2].set(-half)
            # leg mass ~ length (cross-section fixed), times overall mass_scale
            m = self._leg_mass0 * s * mass_scale
            bm = bm.at[bid].set(m)
            bip = bip.at[bid, 2].set(-half)
            # box inertia about COM: local X=thickness,Y=width,Z=length(2*half)
            a, b, c = 2 * self._leg_hx, 2 * self._leg_hy, 2 * half
            ixx = m / 12.0 * (b * b + c * c)
            iyy = m / 12.0 * (a * a + c * c)
            izz = m / 12.0 * (a * a + b * b)
            bi = bi.at[bid].set(jnp.array([ixx, iyy, izz]))
            return gs, gp, bm, bi, bip

        gs, gp = self.sys.geom_size, self.sys.geom_pos
        bm, bi, bip = new_mass, new_inertia, self.sys.body_ipos
        gs, gp, bm, bi, bip = scale_leg((gs, gp, bm, bi, bip), self._legR_gid, self._legR_bid, legR_scale)
        gs, gp, bm, bi, bip = scale_leg((gs, gp, bm, bi, bip), self._legL_gid, self._legL_bid, legL_scale)

        # CoM shift on torso
        bip = bip.at[self._torso_bid].set(jnp.array(self._torso_ipos0) + dcom)

        new_gainprm = self.sys.actuator_gainprm * gain_mult
        new_biasprm = self.sys.actuator_biasprm * gain_mult
        new_friction = self.sys.geom_friction.at[:, 0].set(fric)

        # update geom_rbound for scaled leg boxes (broadphase safety)
        rb = self.sys.geom_rbound
        rR = jnp.sqrt(self._leg_hx**2 + self._leg_hy**2 + (self._leg_half_len0 * legR_scale) ** 2)
        rL = jnp.sqrt(self._leg_hx**2 + self._leg_hy**2 + (self._leg_half_len0 * legL_scale) ** 2)
        rb = rb.at[self._legR_gid].set(rR).at[self._legL_gid].set(rL)

        rsys = self.sys.tree_replace({
            "body_mass": bm, "body_inertia": bi, "body_ipos": bip,
            "geom_size": gs, "geom_pos": gp, "geom_rbound": rb,
            "geom_friction": new_friction,
            "actuator_gainprm": new_gainprm, "actuator_biasprm": new_biasprm,
        })

        latents = {
            "mass_scale": mass_scale, "dcom": dcom,
            "legR_scale": legR_scale, "legL_scale": legL_scale,
            "gain_mult": gain_mult, "fric_mean": jnp.mean(fric),
            "slew": slew, "tau": tau, "cloud_delay": cloud_delay, "imu_offset": imu_offset,
        }
        return rsys, latents

    def reset(self, rng):
        rng, r1, r2, rdr = jax.random.split(rng, 4)
        rsys, lat = self._randomize(rdr)

        qpos = self.sys.qpos0 + jax.random.uniform(r1, (self.sys.nq,), minval=-0.05, maxval=0.05)
        qvel = jax.random.uniform(r2, (self.sys.nv,), minval=-0.05, maxval=0.05)
        data = self.pipeline_init(qpos, qvel)

        obs_hist = jnp.zeros((HISTORY_LEN, FRAME_DIM))
        action_hist = jnp.zeros((5, 2))
        delay_queue = jnp.zeros((5, 2))

        rng, orng = jax.random.split(rng)
        obs, obs_hist = self._get_obs(data, obs_hist, jnp.zeros(2), lat, orng)

        zero = jnp.zeros(())
        metrics = {k: zero for k in [
            "reward_forward", "penalty_ctrl", "penalty_action_rate",
            "x_velocity", "base_height", "upright", "abs_pitch", "abs_roll",
            "action_saturation", "fell", "x_position"]}

        info = {
            "rng": rng, "sys": rsys, "obs_hist": obs_hist, "action_hist": action_hist,
            "real_motor_pos": jnp.zeros(2), "delay_queue": delay_queue,
            "last_action": jnp.zeros(2), "latents": lat,
        }
        return State(data, obs, zero, zero, metrics, info)

    def step(self, state, action):
        data0 = state.pipeline_state
        info = state.info
        sys = info["sys"]
        lat = info["latents"]

        # --- cloud relay: stall + delay ---
        rng, rstall = jax.random.split(info["rng"])
        is_stall = jax.random.uniform(rstall, ()) < STALL_PROB
        dq = info["delay_queue"]
        net_action = jnp.where(is_stall, dq[0], action)
        dq = jnp.concatenate([net_action[None], dq[:-1]])
        delay_idx = lat["cloud_delay"] / self.dt
        i0 = jnp.floor(delay_idx).astype(jnp.int32)
        rem = delay_idx - i0
        delayed = (1.0 - rem) * dq[i0] + rem * dq[i0 + 1]

        # --- actuator slew + 1-pole lag ---
        rmp = info["real_motor_pos"]
        max_delta = lat["slew"] * self.dt
        slewed = jnp.clip(delayed, rmp - max_delta, rmp + max_delta)
        alpha = self.dt / (lat["tau"] + self.dt)
        real_action = (1.0 - alpha) * rmp + alpha * slewed

        # --- physics with per-episode randomized sys ---
        # Drive brax's own mjx pipeline step (type-safe: it reformats contacts)
        # with the per-episode randomized `sys` instead of self.sys.
        def phys(d, _):
            return self._pipeline.step(sys, d, real_action, self._debug), None
        data, _ = jax.lax.scan(phys, data0, (), self._n_frames)

        # --- reward (DR_25hz core: forward vel - ctrl - action_rate) ---
        x_vel = (data.qpos[0] - data0.qpos[0]) / self.dt
        reward_forward = 5.0 * x_vel
        ctrl_cost = 0.02 * jnp.sum(jnp.square(action))
        action_rate = 0.01 * jnp.sum(jnp.square(action - info["last_action"]))
        reward = reward_forward - ctrl_cost - action_rate

        # --- diagnostics ---
        w, x, y, z = data.qpos[3], data.qpos[4], data.qpos[5], data.qpos[6]
        roll = jnp.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = jnp.arcsin(jnp.clip(2 * (w * y - z * x), -1, 1))
        upright = 1 - 2 * (x * x + y * y)
        base_h = data.qpos[2]
        nominal_h = self._leg_half_len0 * 2 * 0.5 * (lat["legL_scale"] + lat["legR_scale"])
        fell = jnp.float32(base_h < 0.4 * nominal_h)
        sat = jnp.mean((jnp.abs(action) > 0.95 * 1.57).astype(jnp.float32))

        rng, orng = jax.random.split(rng)
        new_ahist = jnp.concatenate([action[None], info["action_hist"][:-1]])
        obs, new_obs_hist = self._get_obs(data, info["obs_hist"], action, lat, orng)

        state.metrics.update(
            reward_forward=reward_forward, penalty_ctrl=ctrl_cost,
            penalty_action_rate=action_rate, x_velocity=x_vel, base_height=base_h,
            upright=upright, abs_pitch=jnp.abs(pitch), abs_roll=jnp.abs(roll),
            action_saturation=sat, fell=fell, x_position=data.qpos[0],
        )
        new_info = dict(info)
        new_info.update(rng=rng, action_hist=new_ahist, real_motor_pos=real_action,
                        delay_queue=dq, last_action=action, obs_hist=new_obs_hist)
        return state.replace(pipeline_state=data, obs=obs, reward=reward, done=jnp.zeros(()), info=new_info)

    def _proprio_frame(self, data, last_action, lat, rng):
        w, x, y, z = data.qpos[3], data.qpos[4], data.qpos[5], data.qpos[6]
        roll = jnp.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = jnp.arcsin(jnp.clip(2 * (w * y - z * x), -1, 1))
        yaw = jnp.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))
        ang = jnp.array([roll, pitch, yaw]) + lat["imu_offset"]
        gyro = data.qvel[3:6]
        ra, rg = jax.random.split(rng)
        ang = ang + jax.random.normal(ra, (3,)) * 0.05
        gyro = gyro + jax.random.normal(rg, (3,)) * 0.1
        return jnp.concatenate([ang, gyro, last_action])

    def _get_obs(self, data, obs_hist, last_action, lat, rng):
        frame = self._proprio_frame(data, last_action, lat, rng)
        new_hist = jnp.concatenate([frame[None], obs_hist[:-1]])
        state_obs = new_hist.reshape(-1)  # non-privileged policy obs

        # privileged obs (critic only): clean state + true latents
        w, x, y, z = data.qpos[3], data.qpos[4], data.qpos[5], data.qpos[6]
        roll = jnp.arctan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
        pitch = jnp.arcsin(jnp.clip(2 * (w * y - z * x), -1, 1))
        yaw = jnp.arctan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))
        priv = jnp.concatenate([
            jnp.array([roll, pitch, yaw]), data.qvel[3:6], data.qvel[0:3],
            jnp.array([data.qpos[2]]),
            jnp.array([lat["mass_scale"]]), lat["dcom"],
            jnp.array([lat["legR_scale"], lat["legL_scale"], lat["gain_mult"], lat["fric_mean"]]),
            jnp.mean(lat["slew"], keepdims=True), jnp.mean(lat["tau"], keepdims=True),
            jnp.array([lat["cloud_delay"]]), lat["imu_offset"],
        ])
        return {"state": state_obs, "privileged": priv}, new_hist


envs.register_environment("growbot_rma", GrowbotRMAEnv)

## 5 · Sanity check (auto-runs)
Confirms the model loads, the observation split is right, physics is finite, and leg-length randomization really changes the body — so a broken change fails here, loudly, before the multi-hour train.

In [ ]:
env = envs.get_environment("growbot_rma")
_r = jax.jit(env.reset); _s = jax.jit(env.step)
_st = _r(jax.random.PRNGKey(0))
print("policy obs (non-privileged):", _st.obs["state"].shape,
      "| critic obs (privileged):", _st.obs["privileged"].shape,
      "| actions:", env.action_size)
_st = _s(_st, jnp.zeros(env.action_size))
assert bool(jnp.isfinite(_st.obs["state"]).all()), "non-finite obs!"
print("step OK, reward=%.3f" % float(_st.reward))
print("logged diagnostics:", sorted(_st.metrics.keys()))
print("\nSanity check passed — starting setup for training.")

## 6 · Output dir + TensorBoard writer
Checkpoints + logs go to Google Drive if you allow it (survives disconnects → you can resume), otherwise to local `/content`. Open the TensorBoard panel below to watch training live.

In [ ]:
import os, datetime
from tensorboardX import SummaryWriter

RUN_NAME = "growbot_DR_RMA_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = f"/content/drive/MyDrive/Growbot/{RUN_NAME}"
except Exception as e:
    print("Drive not mounted (", e, ") -> using local /content (lost on disconnect).")
    BASE = f"/content/{RUN_NAME}"

LOG_DIR = f"{BASE}/logs"; CKPT_DIR = f"{BASE}/checkpoints"
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
writer = SummaryWriter(LOG_DIR)
print("Logging to:", LOG_DIR)

In [ ]:
%tensorboard --logdir $LOG_DIR

## 7 · Logging & checkpoint callbacks
Brax reports each episode metric as a **sum over the episode's steps**. That's what we want for the reward (it's the return), but for the diagnostics it would give e.g. "number of fallen steps" instead of a fall *rate* — so those are divided by the episode length here to become clean per-step means (fall fraction, mean speed, mean tilt…). `progress` also prints a one-line auto-diagnosis each eval, and `save_ckpt` checkpoints every eval.

In [ ]:
from brax.io import model
DT = float(env.dt)   # control timestep (0.04 s at 25 Hz)

# episode-total metrics (returns) -> log as-is
SUM_TAGS = {
    "eval/episode_reward": "Reward/1_Total",
    "eval/episode_reward_forward": "Reward/2_Forward",
    "eval/episode_penalty_ctrl": "Penalty/Ctrl_total",
    "eval/episode_penalty_action_rate": "Penalty/ActionRate_total",
}
# per-step diagnostics -> divide by episode length to get a mean/rate
MEAN_TAGS = {
    "eval/episode_x_velocity": "Diagnostics/FwdVelocity_mps",
    "eval/episode_base_height": "Diagnostics/BaseHeight_m",
    "eval/episode_upright": "Diagnostics/Upright_cos",
    "eval/episode_abs_pitch": "Diagnostics/AbsPitch_rad",
    "eval/episode_abs_roll": "Diagnostics/AbsRoll_rad",
    "eval/episode_action_saturation": "Diagnostics/ActionSaturation",
    "eval/episode_fell": "Diagnostics/FallRate",
}
TRAIN_TAGS = {
    "eval/avg_episode_length": "Train/EvalEpisodeLen",
    "training/entropy_loss": "Train/EntropyLoss",
    "training/policy_loss": "Train/PolicyLoss",
    "training/v_loss": "Train/ValueLoss",
    "training/total_loss": "Train/TotalLoss",
    "training/sps": "Train/StepsPerSec",
}

def progress(step, metrics):
    ep_len = float(metrics.get("eval/avg_episode_length", EPISODE_LENGTH)) or EPISODE_LENGTH
    logged = set()
    for k, tag in SUM_TAGS.items():
        if k in metrics: writer.add_scalar(tag, float(metrics[k]), step); logged.add(k)
    for k, tag in MEAN_TAGS.items():
        if k in metrics: writer.add_scalar(tag, float(metrics[k]) / ep_len, step); logged.add(k)
    for k, tag in TRAIN_TAGS.items():
        if k in metrics: writer.add_scalar(tag, float(metrics[k]), step); logged.add(k)
    # net forward displacement over the episode = sum(vel)*dt
    if "eval/episode_x_velocity" in metrics:
        writer.add_scalar("Diagnostics/DistanceTravelled_m", float(metrics["eval/episode_x_velocity"]) * DT, step)
    # keep everything else (incl. *_std spreads) so nothing is lost
    for k, v in metrics.items():
        if k in logged: continue
        try: val = float(v)
        except Exception: continue
        writer.add_scalar("Misc/" + k.replace("/", "_"), val, step)
    writer.flush()

    rew = float(metrics.get("eval/episode_reward", float("nan")))
    vel = float(metrics.get("eval/episode_x_velocity", 0.0)) / ep_len
    fell = float(metrics.get("eval/episode_fell", 0.0)) / ep_len
    sat = float(metrics.get("eval/episode_action_saturation", 0.0)) / ep_len
    if   fell > 0.5:                  note = "HIGH FALL RATE - losing balance (CoM/leg spread too hard, or needs more steps)"
    elif vel < -0.002:               note = "WALKING BACKWARD - heading/reward not learned yet"
    elif vel < 0.005 and fell < 0.2: note = "stable but barely moving - too timid / underpowered / early"
    elif sat > 0.7:                  note = "ACTIONS SATURATING - servo-limited across the mass range"
    else:                             note = "learning to walk"
    print(f"step {step:>10,} | reward {rew:8.1f} | fwd {vel:+.4f} m/s | fall {fell:4.2f} | sat {sat:4.2f} | {note}")

def save_ckpt(step, make_policy, params):
    model.save_params(f"{CKPT_DIR}/step_{step}.pkl", params)

## 8 · Train
Asymmetric actor-critic: `policy_obs_key="state"` (non-privileged, deploys) vs `value_obs_key="privileged"` (latents, training only). This is the multi-hour cell — watch the TensorBoard above.

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=POLICY_HIDDEN,
    value_hidden_layer_sizes=VALUE_HIDDEN,
    policy_obs_key="state",       # non-privileged -> this is what deploys
    value_obs_key="privileged",   # latents -> critic only, discarded at deployment
)

# To resume from a checkpoint, uncomment:
# restore = model.load_params(f"{CKPT_DIR}/step_XXXX.pkl")

print("Training… first eval appears after JIT compile (a few minutes).")
t0 = time.time()
make_inference_fn, params, _ = ppo.train(
    environment=env,
    num_timesteps=NUM_TIMESTEPS,
    num_evals=NUM_EVALS,
    episode_length=EPISODE_LENGTH,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=UNROLL_LENGTH,
    num_minibatches=NUM_MINIBATCHES,
    num_updates_per_batch=NUM_UPDATES_PER_BATCH,
    discounting=DISCOUNTING,
    learning_rate=LEARNING_RATE,
    entropy_cost=ENTROPY_COST,
    reward_scaling=REWARD_SCALING,
    num_envs=NUM_ENVS,
    batch_size=BATCH_SIZE,
    network_factory=network_factory,
    progress_fn=progress,
    policy_params_fn=save_ckpt,
    seed=SEED,
    # restore_params=restore,   # <- and uncomment this to resume
)
print(f"\nDone in {(time.time()-t0)/60:.1f} min.")

## 9 · Save the final policy

In [ ]:
FINAL = f"{BASE}/growbot_DR_RMA_final.pkl"
model.save_params(FINAL, params)
print("Saved:", FINAL)
print("\nDeploy note: the policy consumes ONLY obs['state'] (the non-privileged history).")
print("On the robot, build that same 10-frame stack of [roll,pitch,yaw, gyro_xyz, last_action(2)]")
print("from the phone IMU + last command, normalize with the saved obs stats, and run the policy at 25 Hz.")

## How to read these logs

Every metric is averaged over the evaluation episodes (which span the **full** randomized body distribution), so the curves tell you how the policy copes across *all* builds, not one lucky one.

| TensorBoard tag | Healthy trend | If it goes wrong |
|---|---|---|
| `Reward/1_Total` | rises, then plateaus | flat near 0 → not learning; collapses → instability (see ValueLoss) |
| `Reward/2_Forward` | rises | dominates while Total is low → control penalties too high |
| `Diagnostics/FwdVelocity_mps` | climbs to a steady positive speed | ≈0 → too timid/underpowered · **<0 → walking backward** |
| `Diagnostics/DistanceTravelled_m` | grows | — |
| `Diagnostics/FallRate` | drops toward 0 | stays high → CoM/leg-length spread is too aggressive, or needs more steps |
| `Diagnostics/BaseHeight_m` | steady (matches leg length) | drifts down → collapsing into a belly crawl |
| `Diagnostics/Upright_cos` | near 1 | low → tipping; pair with AbsPitch/AbsRoll to see which way |
| `Diagnostics/AbsPitch_rad` / `AbsRoll_rad` | small | large pitch → nose-diving/rearing; large roll → rolling sideways |
| `Diagnostics/ActionSaturation` | low–moderate | near 1 → servos hitting limits; the heavy tail of the **mass** range may be infeasible — narrow `MASS_SCALE` |
| `Penalty/Ctrl`, `Penalty/ActionRate` | small vs Forward | large → jittery gait; raise the penalty or lower it if it's suppressing motion |
| `Train/ValueLoss` | decreases/stable | **exploding → the critic can't fit the DR spread**: narrow the ranges, or lengthen training |
| `Train/EntropyLoss` | slowly less negative | collapses fast → premature convergence; drops too slowly → stuck exploring |
| `StdDev/*` | shrinks over training | large late → the policy is inconsistent across bodies (some builds it can't handle — inspect which range) |

**Tuning loop:** if `FallRate` stays high and `ValueLoss` is large, the randomization is too hard — narrow `LEG_SHARED` / `MASS_SCALE` / `DCOM_*` in the config cell and rerun. If everything is stable but slow, raise `NUM_TIMESTEPS`. The earlier `DR_25hz` analysis showed this morphology needs tens of millions of steps and keeps improving to ~80M, so a flat early curve is normal — give it time before narrowing ranges.

## Resuming after a disconnect
Checkpoints are in `…/checkpoints/step_*.pkl` on your Drive. To continue: in the **Train** cell, set `restore = model.load_params(".../step_XXXX.pkl")` and uncomment `restore_params=restore`, then run from that cell.

## Extending to full two-phase RMA (optional)
This notebook does concurrent/implicit adaptation (history-conditioned policy + privileged critic). For explicit RMA you would: (1) train a policy that takes the *true* latents through a small encoder, then (2) freeze it and train a history→latent adapter by regression, and deploy policy∘adapter. The env already exposes both the history (`obs['state']`) and the ground-truth latents (`obs['privileged']`) needed for that second phase.